# Imports

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from typing import Optional

import pandas as pd
from openai import OpenAI
from pydantic import BaseModel
from qdrant_client import QdrantClient, models



# Load the CSV

In [ ]:
CSV_PATH = "data/real_estate_medium_table_150_listings.csv"

df = pd.read_csv(CSV_PATH)

print(df.head())

In [ ]:
def row_to_text(row):
    return (
        f"Property: {row['Property_Type']}. "
        f"Location: {row['Neighborhood']}, {row['City']}. "
        f"{int(row['Bedrooms'])} bedrooms and "
        f"{row['Bathrooms']} bathrooms. "
        f"Price: ${row['Price']:,.0f}. "
        f"Approximately {int(row['Square_Feet'])} square feet. "
        f"Built in {int(row['Year_Built'])}. "
        f"Backyard: {'Yes' if row['Has_Backyard'] else 'No'}. "
        f"{row['Description']}"
    )

df["text"] = df.apply(row_to_text, axis=1)

print(df["text"].iloc[0])

# Create Dense + Sparse models

In [ ]:
from sentence_transformers import SentenceTransformer

dense_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
from fastembed import SparseTextEmbedding

sparse_model = SparseTextEmbedding(
    model_name="Qdrant/bm25"
)

# Generate vectors

In [ ]:
dense_vectors = dense_model.encode(
    df["text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

In [ ]:
sparse_vectors = list(
    sparse_model.embed(df["text"].tolist())
)

# Setup Qdrant

## Create Qdrant collection

In [ ]:

client = QdrantClient(
    host="localhost",
    port=6333
)

COLLECTION_NAME = "real_estate_hybrid"

In [ ]:
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,

    vectors_config={
        "dense": models.VectorParams(
            size=384,
            distance=models.Distance.COSINE
        )
    },

    sparse_vectors_config={
        "sparse": models.SparseVectorParams(
            modifier=models.Modifier.IDF
        )
    }
)

## Insert both vectors + metadata

In [ ]:
points = []

for idx, row in df.iterrows():

    sparse = sparse_vectors[idx]

    payload = {
        "listing_id": row["Listing_ID"],
        "price": float(row["Price"]),
        "bedrooms": int(row["Bedrooms"]),
        "bathrooms": float(row["Bathrooms"]),
        "city": row["City"],
        "neighborhood": row["Neighborhood"],
        "property_type": row["Property_Type"],
        "square_feet": int(row["Square_Feet"]),
        "year_built": int(row["Year_Built"]),
        "has_backyard": bool(row["Has_Backyard"]),
        "description": row["Description"],
        "text": row["text"]
    }

    points.append(
        models.PointStruct(
            id=idx,

            vector={
                "dense": dense_vectors[idx].tolist(),

                "sparse": models.SparseVector(
                    indices=sparse.indices.tolist(),
                    values=sparse.values.tolist()
                )
            },

            payload=payload
        )
    )

client.upsert(
    collection_name=COLLECTION_NAME,
    points=points
)

print(f"Inserted {len(points)} listings")

# Hybrid Search

In [ ]:
def hybrid_search(query, limit=5, candidate_limit=20):
    dense_query = dense_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    sparse_query = next(
        sparse_model.embed([query])
    )

    results = client.query_points(
        collection_name=COLLECTION_NAME,

        prefetch=[
            models.Prefetch(
                query=dense_query,
                using="dense",
                limit=candidate_limit
            ),

            models.Prefetch(
                query=models.SparseVector(
                    indices=sparse_query.indices.tolist(),
                    values=sparse_query.values.tolist()
                ),
                using="sparse",
                limit=candidate_limit
            )
        ],

        query=models.FusionQuery(
            fusion=models.Fusion.RRF
        ),

        limit=limit,
        with_payload=True
    ).points

    return results

In [25]:
query = "3-bedroom homes near downtown Austin under 600k"

results = hybrid_search(query)

for result in results:
    print(
        result.payload["listing_id"],
        result.payload["price"],
        result.payload["bedrooms"],
        result.payload["city"],
        result.payload["has_backyard"]
    )

ATX-10001 420000.0 2 Austin True
ATX-10115 557500.0 5 Austin False
ATX-10073 462000.0 4 Austin True
ATX-10097 690000.0 3 Austin True
ATX-10134 483750.0 3 Round Rock False


# Hybrid search with metadata filtering

In [ ]:
def hybrid_search_with_filter(
    query,
    query_filter=None,
    limit=5,
    candidate_limit=20
):

    dense_query = dense_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    sparse_query = next(
        sparse_model.embed([query])
    )

    sparse_vector = models.SparseVector(
        indices=sparse_query.indices.tolist(),
        values=sparse_query.values.tolist()
    )

    results = client.query_points(
        collection_name=COLLECTION_NAME,

        prefetch=[
            models.Prefetch(
                query=dense_query,
                using="dense",
                limit=candidate_limit,
                filter=query_filter
            ),

            models.Prefetch(
                query=sparse_vector,
                using="sparse",
                limit=candidate_limit,
                filter=query_filter
            )
        ],

        query=models.FusionQuery(
            fusion=models.Fusion.RRF
        ),

        limit=limit,
        with_payload=True
    ).points

    return results

In [ ]:
llm_client = OpenAI()


class RealEstateQuery(BaseModel):
    city: Optional[str] = None
    bedrooms: Optional[int] = None
    max_price: Optional[float] = None
    min_price: Optional[float] = None
    has_backyard: Optional[bool] = None
    property_type: Optional[str] = None
    semantic_query: str


def extract_query_metadata(user_query):

    response = llm_client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": """
You extract structured filters from real-estate search queries.

Extract only information explicitly mentioned or clearly implied
by the user's query.

The semantic_query should contain the remaining natural-language
search intent after removing the structured filters.
"""
            },
            {
                "role": "user",
                "content": user_query
            }
        ],
        response_format=RealEstateQuery
    )

    return response.choices[0].message.parsed

In [ ]:
def build_qdrant_filter(filters):

    conditions = []

    if filters.city is not None:
        conditions.append(
            models.FieldCondition(
                key="city",
                match=models.MatchValue(
                    value=filters.city
                )
            )
        )

    if filters.bedrooms is not None:
        conditions.append(
            models.FieldCondition(
                key="bedrooms",
                match=models.MatchValue(
                    value=filters.bedrooms
                )
            )
        )

    if filters.max_price is not None:
        conditions.append(
            models.FieldCondition(
                key="price",
                range=models.Range(
                    lte=filters.max_price
                )
            )
        )

    if filters.min_price is not None:
        conditions.append(
            models.FieldCondition(
                key="price",
                range=models.Range(
                    gte=filters.min_price
                )
            )
        )

    if filters.has_backyard is not None:
        conditions.append(
            models.FieldCondition(
                key="has_backyard",
                match=models.MatchValue(
                    value=filters.has_backyard
                )
            )
        )

    if filters.property_type is not None:
        conditions.append(
            models.FieldCondition(
                key="property_type",
                match=models.MatchValue(
                    value=filters.property_type
                )
            )
        )

    if not conditions:
        return None

    return models.Filter(
        must=conditions
    )

In [31]:
user_query = "3-bedroom homes near downtown Austin under 200k"


filters = extract_query_metadata(user_query)

print(filters)

city='Austin' bedrooms=3 max_price=200000.0 min_price=None has_backyard=None property_type=None semantic_query='homes near downtown'


In [32]:
metadata_filters = build_qdrant_filter(filters)

In [33]:
metadata_filters

Filter(should=None, min_should=None, must=[FieldCondition(key='city', match=MatchValue(value='Austin'), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None), FieldCondition(key='bedrooms', match=MatchValue(value=3), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None), FieldCondition(key='price', match=None, range=Range(lt=None, gt=None, gte=None, lte=200000.0), geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None)], must_not=None)

In [34]:
results = hybrid_search_with_filter(user_query,metadata_filters)

In [35]:
for result in results:
    print(
        result.payload["listing_id"],
        result.payload["price"],
        result.payload["bedrooms"],
        result.payload["city"],
        result.payload["has_backyard"]
    )